In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/model_with_feedback.pkl
/kaggle/input/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/telco_noisy_feedback_prep.csv
/kaggle/input/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/telco_prep.csv
/kaggle/input/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv


In [10]:
# 1. Forcibly remove the locked, outdated version
!pip uninstall -y bitsandbytes

# 2. Install the strict requirements for the modern SLM architecture
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate

# 3. Programmatically terminate and reboot the kernel to clear the system cache
import IPython
print("Installation forced. Assassinating kernel to refresh cache...")
IPython.Application.instance().kernel.do_shutdown(True)

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Installation forced. Assassinating kernel to refresh cache...


{'status': 'ok', 'restart': True}

In [1]:
# 1. THE SILENCERS
import warnings
warnings.filterwarnings('ignore')

import logging
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# 2. STANDARD IMPORTS
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import gc
import time

print("--- INITIALIZING PHASE 3: NLP & PREPROCESSING ---")

# 3. LOAD & CLEAN TABULAR DATA
df = pd.read_csv('/kaggle/input/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop(columns=['customerID', 'PromptInput'], inplace=True, errors='ignore')

# 4. LOAD QUANTIZED SLM
print("Loading 4-bit Quantized Phi-3-mini onto GPU... (Silently)")
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    quantization_config=bnb_config,
    attn_implementation="eager" 
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 5. DEFINE THE EXTRACTION TOOL
def extract_churn_driver(feedback_text):
    prompt = f"""<|user|>
Analyze the following customer feedback. Identify the primary reason for dissatisfaction.
Respond with ONLY ONE of the following exact words: Pricing, Support, Network, Competitor, or None.
Feedback: "{feedback_text}"<|end|>
<|assistant|>"""
    
    outputs = pipe(
        prompt, 
        max_new_tokens=5, 
        do_sample=False, 
        temperature=0.0,
        return_full_text=False
    )
    return outputs[0]['generated_text'].strip()

# 6. EXECUTE FULL EXTRACTION
print("Starting FULL SLM Extraction on 7,032 rows... (Grab a coffee, this takes ~10-15 mins)")
start_time = time.time()

df['Primary_Complaint'] = df['CustomerFeedback'].apply(extract_churn_driver)

end_time = time.time()
print(f"Extraction Complete! Time taken: {round((end_time - start_time)/60, 2)} minutes.\n")

# 7. CATEGORICAL ENCODING
print("Encoding Categorical Variables for Machine Learning...")

df.drop(columns=['CustomerFeedback'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 
    'PaperlessBilling', 'PaymentMethod', 'Primary_Complaint'
]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 8. SAVE ARTIFACTS
df.to_csv('/kaggle/working/processed_churn_data_raw.csv', index=False)
df_encoded.to_csv('/kaggle/working/processed_churn_data_encoded.csv', index=False)

print("--- PHASE 3 COMPLETE ---")
print(f"Final Encoded Dataset Shape: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")

# 9. PURGE VRAM
del model
del pipe
torch.cuda.empty_cache()
gc.collect()
print("GPU Memory Cleared.")

--- INITIALIZING PHASE 3: NLP & PREPROCESSING ---
Loading 4-bit Quantized Phi-3-mini onto GPU... (Silently)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Starting FULL SLM Extraction on 7,032 rows... (Grab a coffee, this takes ~10-15 mins)
Extraction Complete! Time taken: 38.16 minutes.

Encoding Categorical Variables for Machine Learning...
--- PHASE 3 COMPLETE ---
Final Encoded Dataset Shape: 7032 rows, 46 columns
GPU Memory Cleared.


In [4]:
!pip install -q textblob

import pandas as pd
from textblob import TextBlob

print("--- EXECUTING SURGICAL DATA PATCH: SENTIMENT SCORES ---")

# 1. Load the original raw text data
df_original = pd.read_csv('/kaggle/input/datasets/beatafaron/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv')
df_original['TotalCharges'] = pd.to_numeric(df_original['TotalCharges'], errors='coerce')
df_original.dropna(subset=['TotalCharges'], inplace=True)

# 2. Calculate Sentiment Scores instantly
print("Calculating Sentiment Scores...")
# TextBlob assigns a score from -1.0 (Negative) to 1.0 (Positive)
sentiment_scores = df_original['CustomerFeedback'].astype(str).apply(lambda x: TextBlob(x).sentiment.polarity)

# 3. Load our Phase 3 processed datasets
df_encoded = pd.read_csv('/kaggle/working/processed_churn_data_encoded.csv')
df_raw = pd.read_csv('/kaggle/working/processed_churn_data_raw.csv')

# 4. Inject the new feature into our processed datasets
df_encoded['Sentiment_Score'] = sentiment_scores.values
df_raw['Sentiment_Score'] = sentiment_scores.values

# 5. Overwrite the saved artifacts
df_encoded.to_csv('/kaggle/working/processed_churn_data_encoded.csv', index=False)
df_raw.to_csv('/kaggle/working/processed_churn_data_raw.csv', index=False)

print("Patch Complete! 'Sentiment_Score' successfully added to both datasets.")
print(f"New Encoded Shape: {df_encoded.shape[1]} columns")

--- EXECUTING SURGICAL DATA PATCH: SENTIMENT SCORES ---
Calculating Sentiment Scores...
Patch Complete! 'Sentiment_Score' successfully added to both datasets.
New Encoded Shape: 47 columns


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, average_precision_score, roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4: THE MODELING SHOOTOUT (STEP A: THE COMPLETE BASELINES) ---")

# 1. Load the fully patched dataset (with Sentiment Scores)
df = pd.read_csv('/kaggle/working/processed_churn_data_encoded.csv')

X = df.drop(columns=['Churn'])
y = df['Churn']

# 2. Scale the Numerical Features 
# This is absolutely CRITICAL for KNN and SVM, which rely on spatial distance
scaler = StandardScaler()
num_cols = [col for col in X.columns if X[col].nunique() > 2]
X[num_cols] = scaler.fit_transform(X[num_cols])

# 3. Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'PR_AUC': make_scorer(average_precision_score, response_method='predict_proba'),
    'ROC_AUC': make_scorer(roc_auc_score, response_method='predict_proba'),
    'F1': make_scorer(f1_score)
}

# 4. Initialize ALL 6 Baseline Models from the Blueprint
# Note: KNN and GaussianNB do not have native 'class_weight' parameters, 
# so we will see how they suffer against the imbalanced data compared to the others.
baseline_models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    "SVM (Linear)": SVC(kernel='linear', class_weight='balanced', probability=True, random_state=42)
}

# 5. Execute the Shootout
results_list = []

print("Executing 5-Fold Stratified Cross-Validation on ALL Baseline Models...\n")
for name, model in baseline_models.items():
    print(f"Training {name}...")
    cv_results = cross_validate(model, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    
    results_list.append({
        'Model': name,
        'PR-AUC': np.mean(cv_results['test_PR_AUC']),
        'ROC-AUC': np.mean(cv_results['test_ROC_AUC']),
        'F1-Score': np.mean(cv_results['test_F1'])
    })

# 6. Display the Results
baseline_results_df = pd.DataFrame(results_list).sort_values(by='PR-AUC', ascending=False)
print("\n--- COMPLETE BASELINE RESULTS ---")
print(baseline_results_df.to_string(index=False))

--- PHASE 4: THE MODELING SHOOTOUT (STEP A: THE COMPLETE BASELINES) ---
Executing 5-Fold Stratified Cross-Validation on ALL Baseline Models...

Training Logistic Regression...
Training KNN...
Training Naive Bayes...
Training Decision Tree...
Training Random Forest...
Training SVM (Linear)...

--- COMPLETE BASELINE RESULTS ---
              Model   PR-AUC  ROC-AUC  F1-Score
       SVM (Linear) 0.985303 0.993131  0.933810
      Random Forest 0.985188 0.992854  0.944375
Logistic Regression 0.985054 0.993017  0.930548
                KNN 0.946431 0.974277  0.910662
        Naive Bayes 0.894353 0.943488  0.890956
      Decision Tree 0.862884 0.943089  0.916754


In [9]:
!pip install -q xgboost lightgbm catboost

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, average_precision_score, roc_auc_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4: THE MODELING SHOOTOUT (STEP B: GRADIENT BOOSTING) ---")

# 1. Load BOTH updated datasets
df_encoded = pd.read_csv('/kaggle/working/processed_churn_data_encoded.csv')
df_raw = pd.read_csv('/kaggle/working/processed_churn_data_raw.csv')

# 2. Setup Encoded Data (for XGBoost and LightGBM)
X_enc = df_encoded.drop(columns=['Churn'])
y_enc = df_encoded['Churn']

# THE FIX: Sanitize column names for LightGBM's strict JSON parser
# This replaces any character that isn't a letter, number, or underscore with a clean underscore
X_enc = X_enc.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))

# 3. Setup Raw Data (for CatBoost)
X_raw = df_raw.drop(columns=['Churn'])
y_raw = df_raw['Churn']

cat_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 
    'PaperlessBilling', 'PaymentMethod', 'Primary_Complaint'
]

X_raw[cat_features] = X_raw[cat_features].astype(str)

# 4. Cross-Validation & Metric Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'PR_AUC': make_scorer(average_precision_score, response_method='predict_proba'),
    'ROC_AUC': make_scorer(roc_auc_score, response_method='predict_proba'),
    'F1': make_scorer(f1_score)
}

imbalance_ratio = (len(y_enc) - sum(y_enc)) / sum(y_enc)

# 5. Initialize Gradient Boosting Models
gb_models = {
    "XGBoost": xgb.XGBClassifier(scale_pos_weight=imbalance_ratio, eval_metric='aucpr', random_state=42),
    "LightGBM": lgb.LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)
}

results_list = []

print("Executing 5-Fold Stratified Cross-Validation on Gradient Boosters...\n")

# Train XGBoost and LightGBM
for name, model in gb_models.items():
    print(f"Training {name}...")
    cv_results = cross_validate(model, X_enc, y_enc, cv=skf, scoring=scoring, n_jobs=-1)
    
    results_list.append({
        'Model': name,
        'PR-AUC': np.mean(cv_results['test_PR_AUC']),
        'ROC-AUC': np.mean(cv_results['test_ROC_AUC']),
        'F1-Score': np.mean(cv_results['test_F1'])
    })

# Train CatBoost
print("Training CatBoost (Native Categorical Handling)...")
cat_model = CatBoostClassifier(auto_class_weights='Balanced', cat_features=cat_features, verbose=0, random_state=42)
cv_results_cat = cross_validate(cat_model, X_raw, y_raw, cv=skf, scoring=scoring, n_jobs=-1)

results_list.append({
    'Model': 'CatBoost',
    'PR-AUC': np.mean(cv_results_cat['test_PR_AUC']),
    'ROC-AUC': np.mean(cv_results_cat['test_ROC_AUC']),
    'F1-Score': np.mean(cv_results_cat['test_F1'])
})

# 6. Display the Results
gb_results_df = pd.DataFrame(results_list).sort_values(by='PR-AUC', ascending=False)
print("\n--- GRADIENT BOOSTING RESULTS ---")
print(gb_results_df.to_string(index=False))

--- PHASE 4: THE MODELING SHOOTOUT (STEP B: GRADIENT BOOSTING) ---
Executing 5-Fold Stratified Cross-Validation on Gradient Boosters...

Training XGBoost...
Training LightGBM...
Training CatBoost (Native Categorical Handling)...

--- GRADIENT BOOSTING RESULTS ---
   Model   PR-AUC  ROC-AUC  F1-Score
CatBoost 0.988849 0.994696  0.944764
LightGBM 0.986293 0.993544  0.941217
 XGBoost 0.985399 0.993161  0.940142


In [10]:
!pip install -q pytorch-tabnet

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4: THE MODELING SHOOTOUT (STEP C: TABNET DEEP LEARNING) ---")

# 1. Load the Encoded Dataset
df_encoded = pd.read_csv('/kaggle/working/processed_churn_data_encoded.csv')

X_enc = df_encoded.drop(columns=['Churn'])
# Sanitize columns again just to be perfectly safe
X_enc = X_enc.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))
y_enc = df_encoded['Churn'].values

# 2. Deep Learning STRICTLY requires scaled numerical arrays
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_enc)

# 3. Custom Cross-Validation Setup for PyTorch
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pr_auc_scores = []
roc_auc_scores = []
f1_scores = []

print("Executing 5-Fold Stratified Cross-Validation on Google's TabNet...\n")

# 4. The PyTorch Training Loop
for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y_enc), 1):
    print(f"Training Fold {fold}/5...")
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]
    
    # Initialize Google's TabNet Architecture
    clf = TabNetClassifier(
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', # The sequential attention mechanism
        verbose=0,
        seed=42
    )
    
    # Fit the network with early stopping and native class balancing (weights=1)
    clf.fit(
        X_train=X_train, y_train=y_train,
        eval_set=[(X_val, y_val)],
        eval_name=['valid'],
        eval_metric=['auc'],
        max_epochs=50, 
        patience=10,
        batch_size=256, 
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
        weights=1 
    )
    
    # Extract Probabilities and Predictions
    preds_proba = clf.predict_proba(X_val)[:, 1]
    preds = clf.predict(X_val)
    
    pr_auc_scores.append(average_precision_score(y_val, preds_proba))
    roc_auc_scores.append(roc_auc_score(y_val, preds_proba))
    f1_scores.append(f1_score(y_val, preds))

# 5. Compile and Display Results
results_list = [{
    'Model': 'TabNet',
    'PR-AUC': np.mean(pr_auc_scores),
    'ROC-AUC': np.mean(roc_auc_scores),
    'F1-Score': np.mean(f1_scores)
}]

tabnet_results_df = pd.DataFrame(results_list)
print("\n--- DEEP LEARNING RESULTS ---")
print(tabnet_results_df.to_string(index=False))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00
--- PHASE 4: THE MODELING SHOOTOUT (STEP C: TABNET DEEP LEARNING) ---
Executing 5-Fold Stratified Cross-Validation on Google's TabNet...

Training Fold 1/5...

Early stopping occurred at epoch 21 with best_epoch = 11 and best_valid_auc = 0.993
Training Fold 2/5...

Early stopping occurred at epoch 18 with best_epoch = 8 and best_valid_auc = 0.9926
Training Fold 3/5...

Early stopping occurred at epoch 13 with best_epoch = 3 and best_valid_auc = 0.99386
Training Fold 4/5...

Early stopping occurred at epoch 13 with best_epoch = 3 and best_valid_auc = 0.99405
Training Fold 5/5...

Early stopping occurred at epoch 27 with best_epoch = 17 and best_valid_auc = 0.99391

--- DEEP LEARNING RESULTS ---
 Model   PR-AUC  ROC-AUC  F1-Score
TabNet 0.984955 0.993485  0.922842


In [11]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, average_precision_score, roc_auc_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import pandas as pd
import numpy as np
import warnings
import re
warnings.filterwarnings('ignore')

print("--- PHASE 4: THE MODELING SHOOTOUT (STEP D: THE MVP META-MODEL) ---")

# 1. Load the Encoded Dataset
df_encoded = pd.read_csv('/kaggle/working/processed_churn_data_encoded.csv')
X_enc = df_encoded.drop(columns=['Churn'])
X_enc = X_enc.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))
y_enc = df_encoded['Churn']

# 2. Setup the Top 3 Base Estimators
# We use their best configurations from Step B
imbalance_ratio = (len(y_enc) - sum(y_enc)) / sum(y_enc)

estimators = [
    ('cat', CatBoostClassifier(auto_class_weights='Balanced', verbose=0, random_state=42)),
    ('lgb', lgb.LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)),
    ('xgb', xgb.XGBClassifier(scale_pos_weight=imbalance_ratio, eval_metric='aucpr', random_state=42))
]

# 3. Construct the Stacking Classifier
# Logistic Regression acts as the judge, deciding which model's vote counts the most
stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
    cv=5,
    n_jobs=-1
)

# 4. Final Rigorous Evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'PR_AUC': make_scorer(average_precision_score, response_method='predict_proba'),
    'ROC_AUC': make_scorer(roc_auc_score, response_method='predict_proba'),
    'F1': make_scorer(f1_score)
}

print("Executing 5-Fold Stratified Cross-Validation on the Final Stacking Meta-Model...\n")
print("(This may take a few minutes as it trains all 3 heavyweights internally...)")

cv_results_stack = cross_validate(stacking_clf, X_enc, y_enc, cv=skf, scoring=scoring)

results_list = [{
    'Model': 'Ultimate Stacking MVP',
    'PR-AUC': np.mean(cv_results_stack['test_PR_AUC']),
    'ROC-AUC': np.mean(cv_results_stack['test_ROC_AUC']),
    'F1-Score': np.mean(cv_results_stack['test_F1'])
}]

# 5. Display the Final Boss Results
stack_results_df = pd.DataFrame(results_list)
print("\n--- FINAL MVP RESULTS ---")
print(stack_results_df.to_string(index=False))

--- PHASE 4: THE MODELING SHOOTOUT (STEP D: THE MVP META-MODEL) ---
Executing 5-Fold Stratified Cross-Validation on the Final Stacking Meta-Model...

(This may take a few minutes as it trains all 3 heavyweights internally...)

--- FINAL MVP RESULTS ---
                Model   PR-AUC  ROC-AUC  F1-Score
Ultimate Stacking MVP 0.988025 0.994373  0.939955


In [12]:
import pandas as pd
import re
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print("--- PHASE 4: ARTIFACT EXPORT (STEP E: SERIALIZATION) ---")

# 1. Load the full encoded dataset
df_encoded = pd.read_csv('/kaggle/working/processed_churn_data_encoded.csv')
X_enc = df_encoded.drop(columns=['Churn'])
X_enc = X_enc.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))
y_enc = df_encoded['Churn']

# 2. Fit and Save the Scaler (Crucial for local deployment!)
print("Fitting and serializing the StandardScaler...")
scaler = StandardScaler()

# We only scale continuous columns (ignoring binary 0/1 columns)
num_cols = [col for col in X_enc.columns if X_enc[col].nunique() > 2]
scaler.fit(X_enc[num_cols])

# Export the scaler and the exact list of numerical columns it expects
joblib.dump(scaler, '/kaggle/working/churn_scaler.joblib')
joblib.dump(num_cols, '/kaggle/working/num_cols.joblib') 

# 3. Initialize the exact Stacking Architecture
imbalance_ratio = (len(y_enc) - sum(y_enc)) / sum(y_enc)

estimators = [
    ('cat', CatBoostClassifier(auto_class_weights='Balanced', verbose=0, random_state=42)),
    ('lgb', lgb.LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)),
    ('xgb', xgb.XGBClassifier(scale_pos_weight=imbalance_ratio, eval_metric='aucpr', random_state=42))
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
    cv=5,
    n_jobs=-1
)

# 4. Train the Final Boss on 100% of the data
print("Training the Ultimate MVP on 100% of the dataset... (This takes a moment)")
stacking_clf.fit(X_enc, y_enc)

# 5. Serialize the Meta-Model
print("Serializing the Stacking Classifier...")
joblib.dump(stacking_clf, '/kaggle/working/churn_stacking_model.joblib')

# Save the exact column names expected by the model for the Streamlit app later
joblib.dump(list(X_enc.columns), '/kaggle/working/expected_columns.joblib')

print("\n--- STEP E COMPLETE ---")
print("Artifacts successfully saved to /kaggle/working/:")
print("1. churn_scaler.joblib")
print("2. num_cols.joblib")
print("3. expected_columns.joblib")
print("4. churn_stacking_model.joblib")

--- PHASE 4: ARTIFACT EXPORT (STEP E: SERIALIZATION) ---
Fitting and serializing the StandardScaler...
Training the Ultimate MVP on 100% of the dataset... (This takes a moment)
Serializing the Stacking Classifier...

--- STEP E COMPLETE ---
Artifacts successfully saved to /kaggle/working/:
1. churn_scaler.joblib
2. num_cols.joblib
3. expected_columns.joblib
4. churn_stacking_model.joblib
